# Feature Selection

- amt
- merchant
- category
- unix_time (to be finalized)
- gender
- lat 
- long
- state
- city
- zip
- city-pop
- dob (derived to age)
- trans_date_trans_time (derived to hour, day_of_week, is_weekend)
- job
- merch-lat
- merch-long
- is_fraud 

The keep columns: 
- amt: amount per transaction 
- is_weekday: binary 0 and 1
- hours: 0-23
- merchant
- category
- transactions_per_card
- avg_amt_per_card
- unique_merchants_per_card
- age: 2020
- distance
- gender

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\User\anaconda3\Lib\site-packages\numpy\__init__.py:111: UserWarning: mkl-service package failed to import, therefore Intel(R) MKL initialization ensuring its correct out-of-the box operation under condition when Gnu OpenMP had already been loaded by Python process is not assured. Please install mkl-service package, see http://github.com/IntelPython/mkl-service
  from . import _distributor_init


In [2]:
df1 = pd.read_csv("../data/fraudTest.csv")
df2 = pd.read_csv("../data/fraudTrain.csv")

In [ ]:
# Combine both train and test dataset
df = pd.concat([df1, df2], ignore_index=True)

df.head()

In [4]:
# trans_date_trans_time ->  hour, day_of_week, is_weekend (0/1)
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)


# Delete column 'trans_date_trans_time' 
df = df.drop(['trans_date_trans_time','unix_time'], axis=1)

In [11]:
# New Column Age
df['age'] = df['dob'].apply(lambda x: (2020 - pd.to_datetime(x).year)) 

# Delete dob column
df = df.drop('dob', axis=1)

In [ ]:
df.head()



In [5]:
# Create Column distance from merchant to customer
df['distance'] = ((df['merch_lat'] - df['lat'])**2 + (df['merch_long'] - df['long'])**2)**0.5

# Delete columns : lat, long, state, city, zip, city-pop
df = df.drop(['merch_lat', 'merch_long', 'lat', 'long', 'state', 'city', 'zip', 'city_pop'], axis=1)

In [6]:
# Encode Gender (if not yet done)
df['gender'] = df['gender'].map({'M': 0, 'F': 1})

# Delete Column Gender & Job
df = df.drop(['gender','job'], axis=1)

In [7]:
# Derive cc_num into transactions_per_card, avg_amt_per_card, unique_merchants_per_card
df['transactions_per_card'] = df.groupby('cc_num')['cc_num'].transform('count')
df['avg_amt_per_card'] = df.groupby('cc_num')['amt'].transform('mean')
df['unique_merchants_per_card'] = df.groupby('cc_num')['merchant'].transform('nunique')

# Delete cc_num row
df = df.drop('cc_num', axis=1)

In [8]:
# delete column that aren't important
df = df.drop(['first', 'last', 'street', 'trans_num', 'Unnamed: 0'], axis=1)


In [9]:
# Derive merchangt into: 
# - unique_merchants_per_card: Merchant Fraud Rate
# - merchant_avg_amt: Average transaction amount per merchant
# - merchant_avg_amt: Number of Transactions per Merchant

df['merchant_fraud_rate'] = df.groupby('merchant')['is_fraud'].transform('mean')
df['merchant_avg_amt'] = df.groupby('merchant')['amt'].transform('mean')
df['merchant_txn_count'] = df.groupby('merchant')['amt'].transform('count')

# Delete merchangt column
df = df.drop(columns=['merchant'])

In [10]:
# Derive category into:
# - category_fraud_rate: single numeric value representing its fraud risk percentages
# - category_count

df['category_fraud_rate'] = df.groupby('category')['is_fraud'].transform('mean')
df['category_count'] = df['category'].map(df['category'].value_counts())

# Delete category column
df = df.drop('category', axis=1)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# Select features
features = [
    'amt',
    'hour',
    'day_of_week',
    'is_weekend',
    'distance',
    'transactions_per_card',
    'avg_amt_per_card',
    'unique_merchants_per_card',
    'age',
    'merchant_fraud_rate',
    'merchant_avg_amt',
    'merchant_txn_count',
    'category_fraud_rate',
    'category_count'
]

target = 'is_fraud'

X = df[features]
y = df[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# Get feature importance
importance = model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(feature_importance)

# Plot top 15 feature importances
plt.figure(figsize=(10,6))
plt.barh(feature_importance["Feature"][:15], feature_importance["Importance"][:15])
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
# number of unique values
print("Total unique values in 'merchant':", df['merchant'].nunique())
print()

# list all unique values
print("Unique values:")
print(df['merchant'].unique())
print()

# count occurrences of each value
print("Count of each unique value:")
print(df['merchant'].value_counts())

In [14]:
df.to_csv("../data/fraudData-Feature-Engineered.csv", index=False)